# Stage 12 – Model for the Streamlit app

The app in `app/` lets anyone enter an applicant's details and see the risk score, the decision, what drove it and a path to approval. It has to run on a free server without the 1.6 GB dataset, so this notebook trains a **compact monotonic XGBoost model** on the ~20 fields a person can realistically type in, and saves:

- `app/model.json` – the XGBoost model (a few MB)
- `app/model_meta.json` – feature order, category codes, default values, the approval cut-off and test scores

Categories are encoded with fixed code tables (not target encoding), so the app can rebuild every feature exactly from what the user types.

## 1. Load data

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

warnings.filterwarnings("ignore")
CLEAN_FILE = next((p for p in ["../data/accepted_cleaned.csv",
                               r"D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv"]
                   if os.path.exists(p)), "../data/accepted_cleaned.csv")
APP_DIR = "../app"
os.makedirs(APP_DIR, exist_ok=True)
RANDOM_STATE = 42
SAMPLE_SIZE = 300_000

NUMERIC = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low",
           "revol_util", "revol_bal", "open_acc", "total_acc", "inq_last_6mths",
           "delinq_2yrs", "pub_rec", "mort_acc"]
CATEGORICAL = ["term", "grade", "emp_length", "home_ownership", "verification_status", "purpose"]
print("Using data file:", CLEAN_FILE)

header = pl.read_csv(CLEAN_FILE, n_rows=1, infer_schema_length=0).columns
missing = [c for c in NUMERIC + CATEGORICAL + ["default"] if c not in header]
if missing:
    print("Not in the cleaned file, skipped:", missing)
NUMERIC = [c for c in NUMERIC if c in header]
CATEGORICAL = [c for c in CATEGORICAL if c in header]

raw = pl.read_csv(CLEAN_FILE, columns=NUMERIC + CATEGORICAL + ["default"], infer_schema_length=0)
df = raw.sample(n=min(SAMPLE_SIZE, raw.height), seed=RANDOM_STATE).to_pandas()
del raw
t = df["default"].astype(str).str.lower().map({"1": 1, "0": 0, "true": 1, "false": 0, "1.0": 1, "0.0": 0})
df["default"] = t.fillna(0).astype(int)
for c in NUMERIC:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace("%", "", regex=False), errors="coerce")
for c in CATEGORICAL:
    df[c] = df[c].astype(str).str.strip()
print(df.shape, "| default rate", round(df["default"].mean(), 3))

Using data file: D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv
(300000, 21) | default rate 0.2


## 2. Fixed category codes

In [2]:
ORDERED = {
    "term": ["36 months", "60 months"],
    "grade": list("ABCDEFG"),
    "emp_length": ["< 1 year", "1 year", "2 years", "3 years", "4 years", "5 years",
                   "6 years", "7 years", "8 years", "9 years", "10+ years"],
    "verification_status": ["Not Verified", "Source Verified", "Verified"],
}
CODES = {}
for c in CATEGORICAL:
    if c in ORDERED:
        CODES[c] = {v: i for i, v in enumerate(ORDERED[c])}
    else:  # home_ownership, purpose: most common first
        vals = df[c].value_counts()
        vals = vals[vals >= 50].index.tolist()
        CODES[c] = {v: i for i, v in enumerate(vals)}
    print(c, "->", list(CODES[c])[:12])

def encode(frame):
    X = pd.DataFrame(index=frame.index)
    for c in NUMERIC:
        X[c] = frame[c].astype(float)
    for c in CATEGORICAL:
        X[c + "_code"] = frame[c].map(CODES[c]).astype(float)   # unknown -> NaN, XGBoost handles it
    return X

X_all = encode(df)
y_all = df["default"].to_numpy()
FEATURES = list(X_all.columns)
print("Model features:", FEATURES)

term -> ['36 months', '60 months']
grade -> ['A', 'B', 'C', 'D', 'E', 'F', 'G']
emp_length -> ['< 1 year', '1 year', '2 years', '3 years', '4 years', '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years']
home_ownership -> ['MORTGAGE', 'RENT', 'OWN', 'ANY']
verification_status -> ['Not Verified', 'Source Verified', 'Verified']
purpose -> ['debt_consolidation', 'credit_card', 'home_improvement', 'other', 'major_purchase', 'medical', 'small_business', 'car', 'moving', 'vacation', 'house', 'wedding']
Model features: ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'fico_range_low', 'revol_util', 'revol_bal', 'open_acc', 'total_acc', 'inq_last_6mths', 'delinq_2yrs', 'pub_rec', 'mort_acc', 'term_code', 'grade_code', 'emp_length_code', 'home_ownership_code', 'verification_status_code', 'purpose_code']


## 3. Train the compact monotonic model

In [3]:
RULES = {"int_rate": 1, "installment": 1, "loan_amnt": 1, "dti": 1, "revol_util": 1, "revol_bal": 1,
         "inq_last_6mths": 1, "delinq_2yrs": 1, "pub_rec": 1, "grade_code": 1, "term_code": 1,
         "annual_inc": -1, "fico_range_low": -1}
constraints = "(" + ",".join(str(RULES.get(f, 0)) for f in FEATURES) + ")"

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all)
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.15, random_state=RANDOM_STATE, stratify=y_tr)

model = xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", tree_method="hist",
                          n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8,
                          colsample_bytree=0.8, early_stopping_rounds=50, monotone_constraints=constraints,
                          random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

p_te = model.predict_proba(X_te)[:, 1]
prec, rec, _ = precision_recall_curve(y_te, p_te)
roc, pr = roc_auc_score(y_te, p_te), auc(rec, prec)
cutoff = float(np.quantile(p_te, 0.80))
print(f"App model test ROC AUC: {roc:.4f} | PR AUC: {pr:.4f}")
print(f"Approval cut-off (declines riskiest 20%): {cutoff:.1%}")
print(f"Default rate - approved: {y_te[p_te < cutoff].mean():.1%} | declined: {y_te[p_te >= cutoff].mean():.1%}")

App model test ROC AUC: 0.7150 | PR AUC: 0.3817
Approval cut-off (declines riskiest 20%): 29.8%
Default rate - approved: 15.0% | declined: 39.9%


## 4. Save for the app

In [4]:
# keep only the trees chosen by early stopping, so the app can use every tree in the file
best = model.get_booster()[: model.best_iteration + 1]
best.feature_names = FEATURES
check = best.predict(xgb.DMatrix(X_te, feature_names=FEATURES))
print("Saved model matches trained model:", bool(np.allclose(check, p_te, atol=1e-5)))
best.save_model(os.path.join(APP_DIR, "model.json"))

defaults = {c: float(np.nanmedian(df[c])) for c in NUMERIC}
for c in CATEGORICAL:
    defaults[c] = df[c].mode().iloc[0]
meta = {
    "features": FEATURES,
    "numeric": NUMERIC,
    "categorical": CATEGORICAL,
    "codes": CODES,
    "defaults": defaults,
    "cutoff": cutoff,
    "test_roc_auc": float(roc),
    "test_pr_auc": float(pr),
    "train_rows": int(len(X_fit)),
    "base_default_rate": float(y_all.mean()),
}
with open(os.path.join(APP_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=1)

size_mb = os.path.getsize(os.path.join(APP_DIR, "model.json")) / 1e6
print(f"Saved app/model.json ({size_mb:.1f} MB) and app/model_meta.json")

Saved model matches trained model: True
Saved app/model.json (1.1 MB) and app/model_meta.json
